# Climate-conditioned scenario extension and insurability stress testing

I use this notebook to explore how the dependence and event-generation framework could be connected to future climate states and to a financial-loss/insurability layer.

I deliberately do **not** invent new hydraulic climate multipliers here. The climate states are treated as an external conditioning framework that should ultimately be linked to validated hydraulic scenarios, such as KNMI'23/IPCC-based states and adaptation pathways.

For the financial experiments, I use controlled return-period conditioning and explicitly label the resulting calculations as prototype loss proxies rather than validated portfolio or underwriting models.

The main question I want to test is:

**If the dependence structure between breach locations changes the generated event set, how does that propagate into financial loss distributions, and how might the same framework later be conditioned on future climate states?**


In [ ]:

# Notebook 06 — imports and numerical setup


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


RANDOM_SEED = 2026

np.random.seed(RANDOM_SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.2.4
Pandas: 2.2.3


In [ ]:

# KNMI'23 climate-state framework


climate_states = pd.DataFrame({
    "climate_state": [
        "Hd",
        "Hn",
        "Ld",
        "Ln"
    ],
    "emissions": [
        "High",
        "High",
        "Low",
        "Low"
    ],
    "precipitation_storyline": [
        "Dry",
        "Wet",
        "Dry",
        "Wet"
    ],
    "source": [
        "KNMI'23",
        "KNMI'23",
        "KNMI'23",
        "KNMI'23"
    ]
})

display(climate_states)

,climate_state,emissions,precipitation_storyline,source
0,Hd,High,Dry,KNMI'23
1,Hn,High,Wet,KNMI'23
2,Ld,Low,Dry,KNMI'23
3,Ln,Low,Wet,KNMI'23


In [7]:

# Check what persisted project data are available


ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", 'found')
print("Processed data directory:",'found')

if PROCESSED_DIR.exists():
    print("\nFiles:")
    for p in sorted(PROCESSED_DIR.iterdir()):
        print(" -", p.name)
else:
    print("Processed directory not found.")

Project root: found
Processed data directory: found

Files:
 - liwo_scenario_catalog.csv
 - liwo_scenarios.csv


In [ ]:

# Notebook 06 — load processed LIWO data


SCENARIO_FILE = PROCESSED_DIR / "liwo_scenarios.csv"
CATALOG_FILE = PROCESSED_DIR / "liwo_scenario_catalog.csv"

# Load the original scenario catalogue
df_scenarios = pd.read_csv(SCENARIO_FILE)

# Load the merged scenario/location catalogue
df_catalog = pd.read_csv(CATALOG_FILE)

print("Scenario rows:", len(df_scenarios))
print("Catalog rows:", len(df_catalog))

print("\nScenario columns:")
print(df_scenarios.columns.tolist())

print("\nCatalog columns:")
print(df_catalog.columns.tolist())

Scenario rows: 1856
Catalog rows: 1856

Scenario columns:
['breach_id', 'breach_location_id', 'scenario_id', 'scenario_name', 'return_period', 'discharge_m3s', 'breach_name', 'dijkring', 'river', 'damage_MEUR', 'victims', 'affected', 'model_resolution', 'initial_breach_width', 'max_breach_width', 'breach_growth_method', 'scenario_layer', 'map_id']

Catalog columns:
['breach_id', 'breach_location_id', 'scenario_id', 'scenario_name', 'return_period', 'discharge_m3s', 'breach_name', 'dijkring', 'river', 'damage_MEUR', 'victims', 'affected', 'model_resolution', 'initial_breach_width', 'max_breach_width', 'breach_growth_method', 'scenario_layer', 'map_id', 'id', 'breachtypes_id', 'dijkringareas_id', 'openwaters_id', 'name', 'code', 'x', 'y', 'notify', 'lt30', 'f30t300', 'f300t3000', 'f3000t30k', 'gt30k', 'dreigende_overstroming', 'geometry_type', 'geometry_x', 'geometry_y']


In [ ]:

# Clean fields needed for financial-risk analysis


df = df_catalog.copy()

# Numeric damage
df["damage_MEUR_num"] = pd.to_numeric(
    df["damage_MEUR"],
    errors="coerce"
)

# Log damage for structural analyses
df["log_damage"] = np.log1p(
    df["damage_MEUR_num"]
)

# Numeric return period
df["return_period"] = pd.to_numeric(
    df["return_period"],
    errors="coerce"
)

print("Rows:", len(df))
print(
    "Usable numeric damage:",
    df["damage_MEUR_num"].notna().sum()
)
print(
    "Usable return periods:",
    df["return_period"].notna().sum()
)

display(
    df[
        [
            "breach_id",
            "return_period",
            "damage_MEUR",
            "damage_MEUR_num",
            "river",
            "dijkring"
        ]
    ].head()
)

Rows: 1856
Usable numeric damage: 1830
Usable return periods: 1856


,breach_id,return_period,damage_MEUR,damage_MEUR_num,river,dijkring
0,1,400000,290,290.0,Oosterschelde,Dijkring 27 - Tholen en St. Philipsland
1,14,125,52,52.0,Maas,Dijkring 39 - Alem
2,14,1250,65,65.0,Maas,Dijkring 39 - Alem
3,14,12500,81,81.0,Maas,Dijkring 39 - Alem
4,25,125,1400,1400.0,Maas,Dijkring 41 - Land van Maas en Waal


In [ ]:

# T4000 location-level damage lookup


t4000 = df[
    df["return_period"].eq(4000)
    & df["damage_MEUR_num"].notna()
].copy()

t4000_damage_lookup = (
    t4000
    .groupby("breach_id")["damage_MEUR_num"]
    .median()
    .to_dict()
)

print(
    "T4000 locations with usable damage:",
    len(t4000_damage_lookup)
)

T4000 locations with usable damage: 268


In [ ]:

# Recreate the provisional handcrafted dependence kernel


# Location table from the saved LIWO catalogue
locations = (
    df_catalog[
        [
            "breach_id",
            "geometry_x",
            "geometry_y",
            "river",
            "dijkring"
        ]
    ]
    .drop_duplicates("breach_id")
    .dropna(subset=["geometry_x", "geometry_y"])
    .copy()
)

locations["breach_id"] = (
    locations["breach_id"]
    .astype(int)
)

locations["river"] = (
    locations["river"]
    .fillna("UNKNOWN")
    .astype(str)
)

locations["dijkring"] = (
    locations["dijkring"]
    .fillna("UNKNOWN")
    .astype(str)
)

print("Kernel locations:", len(locations))


# Selected provisional kernel parameters


KERNEL_RADIUS_KM = 25.0
KERNEL_LENGTH_SCALE_KM = 10.0
KERNEL_RIVER_WEIGHT = 0.5
KERNEL_DIKE_WEIGHT = 1.0

kernel_lookup = {}

loc_rows = locations.to_dict("records")

for a in loc_rows:

    source = int(a["breach_id"])

    targets = []
    raw_weights = []

    for b in loc_rows:

        target = int(b["breach_id"])

        if source == target:
            continue

        dx = a["geometry_x"] - b["geometry_x"]
        dy = a["geometry_y"] - b["geometry_y"]

        distance_km = (
            np.sqrt(dx**2 + dy**2) / 1000.0
        )

        if distance_km > KERNEL_RADIUS_KM:
            continue

        same_river = (
            a["river"] == b["river"]
        )

        same_dike = (
            a["dijkring"] == b["dijkring"]
        )

        # Provisional dependence kernel
        weight = np.exp(
            -distance_km
            / KERNEL_LENGTH_SCALE_KM
        )

        if same_river:
            weight *= (
                1.0 + KERNEL_RIVER_WEIGHT
            )

        if same_dike:
            weight *= (
                1.0 + KERNEL_DIKE_WEIGHT
            )

        targets.append(target)
        raw_weights.append(weight)

    if len(targets) > 0:

        targets = np.asarray(
            targets,
            dtype=int
        )

        weights = np.asarray(
            raw_weights,
            dtype=float
        )

        weights = (
            weights / weights.sum()
        )

        kernel_lookup[source] = {
            "targets": targets,
            "weights": weights
        }

print(
    "Kernel sources:",
    len(kernel_lookup)
)

Kernel locations: 619
Kernel sources: 619


In [13]:
first_source = next(iter(kernel_lookup))

print("First source:", first_source)
print("Number of targets:",
      len(kernel_lookup[first_source]["targets"]))

print(
    "Weight sum:",
    kernel_lookup[first_source]["weights"].sum()
)

First source: 1
Number of targets: 131
Weight sum: 1.0000000000000002


In [ ]:

# Load persisted transition models


gnn_transition_df = pd.read_csv(
    PROCESSED_DIR.parent.parent
    / "results"
    / "tables"
    / "gnn_transition_probabilities.csv"
)

kernel_transition_df = pd.read_csv(
    PROCESSED_DIR.parent.parent
    / "results"
    / "tables"
    / "handcrafted_kernel_probabilities.csv"
)

print(
    "GNN transition rows:",
    len(gnn_transition_df)
)

print(
    "GNN source locations:",
    gnn_transition_df["breach_i"].nunique()
)

print(
    "Kernel transition rows:",
    len(kernel_transition_df)
)

print(
    "Kernel source locations:",
    kernel_transition_df["breach_i"].nunique()
)

GNN transition rows: 36393
GNN source locations: 569
Kernel transition rows: 21487
Kernel source locations: 594


In [ ]:

# Reconstruct transition dictionaries


gnn_transition = {}

for source, group in gnn_transition_df.groupby("breach_i"):

    targets = group["breach_j"].astype(int).to_numpy()
    probabilities = group["gnn_probability"].astype(float).to_numpy()

    probabilities = probabilities / probabilities.sum()

    gnn_transition[int(source)] = (
        targets,
        probabilities
    )


kernel_lookup = {}

for source, group in kernel_transition_df.groupby("breach_i"):

    targets = group["breach_j"].astype(int).to_numpy()
    weights = group["kernel_probability"].astype(float).to_numpy()

    weights = weights / weights.sum()

    kernel_lookup[int(source)] = {
        "targets": targets,
        "weights": weights
    }


print(
    "Reconstructed GNN sources:",
    len(gnn_transition)
)

print(
    "Reconstructed kernel sources:",
    len(kernel_lookup)
)

Reconstructed GNN sources: 569
Reconstructed kernel sources: 594


In [ ]:

# Check GNN vs kernel source coverage


kernel_sources = set(
    kernel_transition_df["breach_i"]
    .astype(int)
)

gnn_sources = set(
    gnn_transition_df["breach_i"]
    .astype(int)
)

missing_gnn_sources = sorted(
    kernel_sources - gnn_sources
)

print(
    "Kernel source locations:",
    len(kernel_sources)
)

print(
    "GNN source locations:",
    len(gnn_sources)
)

print(
    "Kernel sources missing from GNN:",
    len(missing_gnn_sources)
)

print(
    "Missing GNN source IDs:",
    missing_gnn_sources
)

Kernel source locations: 594
GNN source locations: 569
Kernel sources missing from GNN: 25
Missing GNN source IDs: [744, 978, 1175, 1176, 1177, 1178, 1179, 1180, 1181, 1182, 1687, 1806, 1807, 1871, 2174, 2283, 2284, 2854, 2932, 2999, 3086, 3099, 3127, 3468, 3470]


In [ ]:

# Diagnose missing GNN source locations


if "gnn_pairs" in globals():

    missing_pair_sources = (
        gnn_pairs[
            gnn_pairs["breach_i"].isin(
                missing_gnn_sources
            )
        ]
        .groupby("breach_i")
        .size()
    )

    print(
        "Missing-source pair counts:"
    )

    display(
        missing_pair_sources
        .reindex(missing_gnn_sources)
    )

else:
    print(
        "gnn_pairs is not available in Notebook 06."
    )

gnn_pairs is not available in Notebook 06.


In [ ]:

# GNN transition source statistics


gnn_source_counts = (
    gnn_transition_df
    .groupby("breach_i")
    .size()
)

print(
    "Min transitions per GNN source:",
    gnn_source_counts.min()
)

print(
    "Median transitions per GNN source:",
    gnn_source_counts.median()
)

print(
    "Max transitions per GNN source:",
    gnn_source_counts.max()
)

display(
    gnn_source_counts.describe()
)

Min transitions per GNN source: 1
Median transitions per GNN source: 39.0
Max transitions per GNN source: 325


count    569.000000
mean      63.959578
std       66.966542
min        1.000000
25%       16.000000
50%       39.000000
75%       93.000000
max      325.000000
dtype: float64

In [ ]:

# Coverage-safe GNN transitions
# Local uniform fallback within 25 km


gnn_t4000_transition = {}

loc_lookup = (
    df_catalog
    .drop_duplicates("breach_id")
    .set_index("breach_id")
)

for source in t4000_sources:

    source = int(source)

    
    # Learned GNN distribution
    

    if source in gnn_transition:

        targets, probabilities = (
            gnn_transition[source]
        )

        targets = np.asarray(
            targets,
            dtype=int
        )

        probabilities = np.asarray(
            probabilities,
            dtype=float
        )

        mask = np.isin(
            targets,
            t4000_sources
        )

        targets = targets[mask]
        probabilities = probabilities[mask]

        if (
            len(targets) > 0
            and probabilities.sum() > 0
        ):

            probabilities /= probabilities.sum()

            gnn_t4000_transition[source] = {
                "targets": targets,
                "probabilities": probabilities,
                "fallback": False
            }

            continue

    
    # Fallback: uniform among T4000 locations within 25 km
    

    if source not in loc_lookup.index:
        continue

    sx = float(
        loc_lookup.loc[
            source,
            "geometry_x"
        ]
    )

    sy = float(
        loc_lookup.loc[
            source,
            "geometry_y"
        ]
    )

    candidate_targets = []
    candidate_distances = []

    for target in t4000_sources:

        target = int(target)

        if target == source:
            continue

        tx = float(
            loc_lookup.loc[
                target,
                "geometry_x"
            ]
        )

        ty = float(
            loc_lookup.loc[
                target,
                "geometry_y"
            ]
        )

        d_km = (
            np.sqrt(
                (sx - tx)**2
                + (sy - ty)**2
            ) / 1000
        )

        if d_km <= 25:

            candidate_targets.append(target)
            candidate_distances.append(d_km)

    if len(candidate_targets) > 0:

        candidate_targets = np.asarray(
            candidate_targets,
            dtype=int
        )

        probabilities = np.ones(
            len(candidate_targets),
            dtype=float
        )

        probabilities /= probabilities.sum()

        gnn_t4000_transition[source] = {
            "targets": candidate_targets,
            "probabilities": probabilities,
            "fallback": True
        }


n_fallback = sum(
    info["fallback"]
    for info in gnn_t4000_transition.values()
)

print(
    "T4000 sources with usable GNN transition:",
    len(gnn_t4000_transition)
)

print(
    "Sources using local uniform fallback:",
    n_fallback
)

T4000 sources with usable GNN transition: 266
Sources using local uniform fallback: 7


In [24]:

# T4000 GNN generator with fallback tracking


def generate_gnn_events_with_tracking(
    transition_lookup,
    sources,
    n_events=5000,
    event_size=3,
    seed=42
):
    rng = np.random.default_rng(seed)

    sources = np.asarray(
        list(sources),
        dtype=int
    )

    events = []
    used_fallback = []

    for _ in range(n_events):

        first = int(rng.choice(sources))
        event = [first]
        fallback_used = False

        while len(event) < event_size:

            current = event[-1]

            if current not in transition_lookup:
                break

            info = transition_lookup[current]

            targets = np.asarray(
                info["targets"],
                dtype=int
            )

            probabilities = np.asarray(
                info["probabilities"],
                dtype=float
            )

            mask = ~np.isin(
                targets,
                np.asarray(event)
            )

            targets = targets[mask]
            probabilities = probabilities[mask]

            if (
                len(targets) == 0
                or probabilities.sum() <= 0
            ):
                break

            probabilities = (
                probabilities
                / probabilities.sum()
            )

            if info.get("fallback", False):
                fallback_used = True

            nxt = int(
                rng.choice(
                    targets,
                    p=probabilities
                )
            )

            event.append(nxt)

        if len(event) == event_size:

            events.append(event)
            used_fallback.append(fallback_used)

    return events, np.asarray(used_fallback, dtype=bool)


t4000_gnn_events, t4000_gnn_fallback = (
    generate_gnn_events_with_tracking(
        gnn_t4000_transition,
        t4000_sources,
        n_events=5000,
        event_size=3,
        seed=4002
    )
)

print("Generated events:", len(t4000_gnn_events))
print(
    "Events using fallback:",
    t4000_gnn_fallback.sum()
)
print(
    "Fallback event share:",
    t4000_gnn_fallback.mean()
)

Generated events: 4957
Events using fallback: 492
Fallback event share: 0.09925358079483558


In [ ]:

# T4000 GNN losses with fallback tracking


t4000_gnn_losses = []

for event in t4000_gnn_events:

    values = [
        t4000_damage_lookup[int(b)]
        for b in event
        if int(b) in t4000_damage_lookup
    ]

    if len(values) == len(event):
        t4000_gnn_losses.append(np.sum(values))

t4000_gnn_losses = np.asarray(
    t4000_gnn_losses
)

print(
    "Usable loss observations:",
    len(t4000_gnn_losses)
)

Usable loss observations: 4957


In [ ]:

# Notebook 06 — restricted handcrafted-kernel event generator


def generate_restricted_kernel_events(
    transition_lookup,
    sources,
    n_events=5000,
    event_size=3,
    seed=42
):
    rng = np.random.default_rng(seed)

    sources = np.asarray(
        list(sources),
        dtype=int
    )

    events = []

    for _ in range(n_events):

        first = int(
            rng.choice(sources)
        )

        event = [first]

        while len(event) < event_size:

            current = event[-1]

            # No transition available
            if current not in transition_lookup:
                break

            info = transition_lookup[current]

            targets = np.asarray(
                info["targets"],
                dtype=int
            )

            weights = np.asarray(
                info["weights"],
                dtype=float
            )

            # Exclude locations already selected
            mask = ~np.isin(
                targets,
                np.asarray(event)
            )

            available_targets = targets[mask]
            available_weights = weights[mask]

            if (
                len(available_targets) == 0
                or available_weights.sum() <= 0
            ):
                break

            available_weights = (
                available_weights
                / available_weights.sum()
            )

            next_location = int(
                rng.choice(
                    available_targets,
                    p=available_weights
                )
            )

            event.append(next_location)

        if len(event) == event_size:
            events.append(event)

    return events



# Generate T4000 kernel events


N_T4000_EVENTS = 5000
EVENT_SIZE = 3

t4000_kernel_events = generate_restricted_kernel_events(
    kernel_t4000_lookup,
    t4000_sources,
    n_events=N_T4000_EVENTS,
    event_size=EVENT_SIZE,
    seed=4001
)

print(
    "T4000 handcrafted-kernel events:",
    len(t4000_kernel_events)
)

print(
    "Example events:",
    t4000_kernel_events[:5]
)

T4000 handcrafted-kernel events: 0
Example events: []


In [ ]:

# Recreate all T4000 loss samples


t4000_loss_ind = event_losses_restricted(
    t4000_independent_events,
    t4000_damage_lookup
)

t4000_loss_kernel = event_losses_restricted(
    t4000_kernel_events,
    t4000_damage_lookup
)

t4000_loss_gnn = event_losses_restricted(
    t4000_gnn_events,
    t4000_damage_lookup
)

print("Independent:", len(t4000_loss_ind))
print("Kernel:", len(t4000_loss_kernel))
print("GNN:", len(t4000_loss_gnn))

Independent: 5000
Kernel: 0
GNN: 4957


In [ ]:

# Diagnose T4000 handcrafted-kernel coverage


print("T4000 locations:", len(t4000_sources))
print("Kernel transition sources:", len(kernel_t4000_lookup))

# Number of T4000 sources with at least one target
usable_kernel_sources = {
    int(source): info
    for source, info in kernel_t4000_lookup.items()
    if len(info["targets"]) > 0
    and np.asarray(info["weights"]).sum() > 0
}

print(
    "Usable kernel sources:",
    len(usable_kernel_sources)
)

# How many targets do they have?
target_counts = np.array([
    len(info["targets"])
    for info in usable_kernel_sources.values()
])

if len(target_counts):
    print(
        "Target count — min / median / max:",
        target_counts.min(),
        np.median(target_counts),
        target_counts.max()
    )

# Show a few sources
for source, info in list(
    usable_kernel_sources.items()
)[:5]:

    print(
        source,
        "targets =", len(info["targets"]),
        "weight_sum =",
        np.asarray(info["weights"]).sum()
    )

T4000 locations: 268
Kernel transition sources: 0
Usable kernel sources: 0


In [ ]:

# Kernel T4000 source/target intersection


kernel_all_sources = set(
    kernel_lookup.keys()
)

print(
    "Kernel total sources:",
    len(kernel_all_sources)
)

print(
    "T4000 ∩ kernel sources:",
    len(
        set(t4000_sources)
        & kernel_all_sources
    )
)

# For each T4000 source, count kernel targets that are T4000
intersection_counts = []

for source in t4000_sources:

    source = int(source)

    if source not in kernel_lookup:
        intersection_counts.append(0)
        continue

    targets = np.asarray(
        kernel_lookup[source]["targets"],
        dtype=int
    )

    intersection_counts.append(
        np.isin(
            targets,
            t4000_sources
        ).sum()
    )

intersection_counts = np.asarray(
    intersection_counts
)

print(
    "T4000 sources with >=1 T4000 kernel target:",
    np.sum(intersection_counts > 0)
)

print(
    "T4000 target count — min / median / max:",
    intersection_counts.min(),
    np.median(intersection_counts),
    intersection_counts.max()
)

Kernel total sources: 594
T4000 ∩ kernel sources: 262
T4000 sources with >=1 T4000 kernel target: 259
T4000 target count — min / median / max: 0 30.0 102


In [34]:

# Build T4000-restricted handcrafted kernel
# directly from the persisted original kernel table


t4000_set = set(
    t4000_sources.astype(int)
)

kernel_t4000_lookup = {}

for source, info in kernel_lookup.items():

    source = int(source)

    # Keep only T4000 source locations
    if source not in t4000_set:
        continue

    targets = np.asarray(
        info["targets"],
        dtype=int
    )

    weights = np.asarray(
        info["weights"],
        dtype=float
    )

    # Keep only T4000 target locations
    mask = np.isin(
        targets,
        list(t4000_set)
    )

    targets_t4000 = targets[mask]
    weights_t4000 = weights[mask]

    # Renormalize after removing non-T4000 targets
    if (
        len(targets_t4000) > 0
        and weights_t4000.sum() > 0
    ):

        weights_t4000 = (
            weights_t4000
            / weights_t4000.sum()
        )

        kernel_t4000_lookup[source] = {
            "targets": targets_t4000,
            "weights": weights_t4000
        }

print(
    "T4000 kernel sources with usable targets:",
    len(kernel_t4000_lookup)
)

# Sanity check
target_counts = np.array([
    len(info["targets"])
    for info in kernel_t4000_lookup.values()
])

print(
    "Target count — min / median / max:",
    target_counts.min(),
    np.median(target_counts),
    target_counts.max()
)

print(
    "Example source:",
    next(iter(kernel_t4000_lookup))
)

T4000 kernel sources with usable targets: 259
Target count — min / median / max: 1 31.0 102
Example source: 385


In [ ]:

# Generate T4000 handcrafted-kernel events


t4000_kernel_events = generate_restricted_kernel_events(
    kernel_t4000_lookup,
    t4000_sources,
    n_events=5000,
    event_size=3,
    seed=4001
)

print(
    "T4000 handcrafted-kernel events:",
    len(t4000_kernel_events)
)

print(
    "Example events:",
    t4000_kernel_events[:5]
)

T4000 handcrafted-kernel events: 4462
Example events: [[3360, 3363, 3364], [1953, 2394, 2395], [385, 2313, 3312], [643, 647, 648], [2328, 2347, 2349]]


In [ ]:

# Robust GNN event generator
# Handles both tuple and dictionary transition formats


def generate_gnn_events_robust(
    transition_lookup,
    sources,
    locations_df,
    n_events=5000,
    event_size=3,
    radius_km=25,
    seed=42
):
    rng = np.random.default_rng(seed)

    sources = np.asarray(
        list(sources),
        dtype=int
    )

    loc = (
        locations_df
        .drop_duplicates("breach_id")
        .set_index("breach_id")
    )

    events = []
    fallback_flags = []

    for _ in range(n_events):

        first = int(rng.choice(sources))

        event = [first]
        used_fallback = False

        while len(event) < event_size:

            current = event[-1]

            # ------------------------------------------------
            # Valid unused locations
            # ------------------------------------------------

            valid_targets = np.array(
                [
                    int(x)
                    for x in sources
                    if int(x) not in event
                ],
                dtype=int
            )

            # ------------------------------------------------
            # Try learned GNN transition
            # ------------------------------------------------

            learned_success = False

            if current in transition_lookup:

                info = transition_lookup[current]

                # Dictionary format
                if isinstance(info, dict):

                    targets = np.asarray(
                        info["targets"],
                        dtype=int
                    )

                    probabilities = np.asarray(
                        info["probabilities"],
                        dtype=float
                    )

                    is_fallback_entry = bool(
                        info.get("fallback", False)
                    )

                # Tuple format
                else:

                    targets = np.asarray(
                        info[0],
                        dtype=int
                    )

                    probabilities = np.asarray(
                        info[1],
                        dtype=float
                    )

                    is_fallback_entry = False

                mask = (
                    np.isin(
                        targets,
                        valid_targets
                    )
                    &
                    (probabilities > 0)
                )

                targets = targets[mask]
                probabilities = probabilities[mask]

                if (
                    len(targets) > 0
                    and probabilities.sum() > 0
                ):

                    probabilities = (
                        probabilities
                        / probabilities.sum()
                    )

                    nxt = int(
                        rng.choice(
                            targets,
                            p=probabilities
                        )
                    )

                    event.append(nxt)

                    if is_fallback_entry:
                        used_fallback = True

                    learned_success = True

            if learned_success:
                continue

            # ------------------------------------------------
            # Local uniform fallback within 25 km
            # ------------------------------------------------

            local_targets = []

            if current in loc.index:

                cx = float(
                    loc.loc[
                        current,
                        "geometry_x"
                    ]
                )

                cy = float(
                    loc.loc[
                        current,
                        "geometry_y"
                    ]
                )

                for target in valid_targets:

                    tx = float(
                        loc.loc[
                            target,
                            "geometry_x"
                        ]
                    )

                    ty = float(
                        loc.loc[
                            target,
                            "geometry_y"
                        ]
                    )

                    distance_km = (
                        np.sqrt(
                            (cx - tx) ** 2
                            + (cy - ty) ** 2
                        )
                        / 1000.0
                    )

                    if distance_km <= radius_km:
                        local_targets.append(
                            int(target)
                        )

            if len(local_targets) > 0:

                nxt = int(
                    rng.choice(local_targets)
                )

                event.append(nxt)
                used_fallback = True

            else:

                # Final fallback: any unused source
                if len(valid_targets) == 0:
                    break

                nxt = int(
                    rng.choice(valid_targets)
                )

                event.append(nxt)
                used_fallback = True

        if len(event) == event_size:

            events.append(event)

            fallback_flags.append(
                used_fallback
            )

    return (
        events,
        np.asarray(
            fallback_flags,
            dtype=bool
        )
    )

In [ ]:

# Generate 5,000 T4000 GNN events


t4000_gnn_events, t4000_gnn_fallback = (
    generate_gnn_events_robust(
        gnn_t4000_transition,
        t4000_sources,
        locations,
        n_events=5000,
        event_size=3,
        radius_km=25,
        seed=4002
    )
)

print(
    "GNN events:",
    len(t4000_gnn_events)
)

print(
    "GNN fallback events:",
    t4000_gnn_fallback.sum()
)

print(
    "GNN fallback share:",
    t4000_gnn_fallback.mean()
)

GNN events: 5000
GNN fallback events: 572
GNN fallback share: 0.1144


In [43]:
print("Independent:", len(t4000_independent_events))
print("Kernel:", len(t4000_kernel_events))
print("GNN:", len(t4000_gnn_events))

Independent: 5000
Kernel: 5000
GNN: 5000


In [ ]:

# T4000 GNN loss: learned-only vs fallback events


gnn_event_losses = []

for event in t4000_gnn_events:

    values = [
        t4000_damage_lookup[int(b)]
        for b in event
    ]

    gnn_event_losses.append(
        np.sum(values)
    )

gnn_event_losses = np.asarray(
    gnn_event_losses,
    dtype=float
)

print("Total GNN events:", len(gnn_event_losses))
print(
    "Fallback events:",
    t4000_gnn_fallback.sum()
)
print(
    "Non-fallback events:",
    (~t4000_gnn_fallback).sum()
)

Total GNN events: 5000
Fallback events: 572
Non-fallback events: 4428


In [ ]:

# Compare GNN loss distributions by fallback usage


gnn_fallback_loss_summary = pd.DataFrame({
    "group": [
        "No fallback",
        "Fallback"
    ],
    "n_events": [
        (~t4000_gnn_fallback).sum(),
        t4000_gnn_fallback.sum()
    ],
    "mean_MEUR": [
        gnn_event_losses[~t4000_gnn_fallback].mean(),
        gnn_event_losses[t4000_gnn_fallback].mean()
    ],
    "median_MEUR": [
        np.median(
            gnn_event_losses[~t4000_gnn_fallback]
        ),
        np.median(
            gnn_event_losses[t4000_gnn_fallback]
        )
    ],
    "P95_MEUR": [
        np.quantile(
            gnn_event_losses[~t4000_gnn_fallback],
            0.95
        ),
        np.quantile(
            gnn_event_losses[t4000_gnn_fallback],
            0.95
        )
    ],
    "P99_MEUR": [
        np.quantile(
            gnn_event_losses[~t4000_gnn_fallback],
            0.99
        ),
        np.quantile(
            gnn_event_losses[t4000_gnn_fallback],
            0.99
        )
    ]
})

display(gnn_fallback_loss_summary)

,group,n_events,mean_MEUR,median_MEUR,P95_MEUR,P99_MEUR
0,No fallback,4428,1722.150632,570.0,8000.0,15393.28
1,Fallback,572,2389.770979,751.0,7294.0,23425.62


In [48]:

# Final corrected T4000 financial comparison


# Independent losses
t4000_loss_ind = event_losses_restricted(
    t4000_independent_events,
    t4000_damage_lookup
)

# Kernel losses
t4000_loss_kernel = event_losses_restricted(
    t4000_kernel_events,
    t4000_damage_lookup
)

# Corrected GNN losses
t4000_loss_gnn = gnn_event_losses

final_t4000_comparison = pd.DataFrame({
    "generator": [
        "Independent",
        "Handcrafted kernel",
        "GNN learned"
    ],
    "n_events": [
        len(t4000_loss_ind),
        len(t4000_loss_kernel),
        len(t4000_loss_gnn)
    ],
    "mean_MEUR": [
        np.mean(t4000_loss_ind),
        np.mean(t4000_loss_kernel),
        np.mean(t4000_loss_gnn)
    ],
    "median_MEUR": [
        np.median(t4000_loss_ind),
        np.median(t4000_loss_kernel),
        np.median(t4000_loss_gnn)
    ],
    "P90_MEUR": [
        np.quantile(t4000_loss_ind, 0.90),
        np.quantile(t4000_loss_kernel, 0.90),
        np.quantile(t4000_loss_gnn, 0.90)
    ],
    "P95_MEUR": [
        np.quantile(t4000_loss_ind, 0.95),
        np.quantile(t4000_loss_kernel, 0.95),
        np.quantile(t4000_loss_gnn, 0.95)
    ],
    "P99_MEUR": [
        np.quantile(t4000_loss_ind, 0.99),
        np.quantile(t4000_loss_kernel, 0.99),
        np.quantile(t4000_loss_gnn, 0.99)
    ],
    "max_MEUR": [
        np.max(t4000_loss_ind),
        np.max(t4000_loss_kernel),
        np.max(t4000_loss_gnn)
    ]
})

display(final_t4000_comparison)

,generator,n_events,mean_MEUR,median_MEUR,P90_MEUR,P95_MEUR,P99_MEUR,max_MEUR
0,Independent,5000,2126.0586,940.5,5331.0,8023.10,23062.0,33400.0
1,Handcrafted kernel,5000,1708.5394,582.0,4500.0,7294.00,15790.0,27901.0
2,GNN learned,5000,1798.5264,601.0,4809.0,7915.05,15790.0,26140.0


In [ ]:

# Fallback contribution to GNN mean loss


overall_mean = gnn_event_losses.mean()

no_fallback_mean = (
    gnn_event_losses[~t4000_gnn_fallback].mean()
)

fallback_mean = (
    gnn_event_losses[t4000_gnn_fallback].mean()
)

print("Overall GNN mean:", overall_mean)
print("No-fallback mean:", no_fallback_mean)
print("Fallback mean:", fallback_mean)

print(
    "Difference in means (%):",
    100 * (
        fallback_mean - no_fallback_mean
    ) / no_fallback_mean
)

Overall GNN mean: 1798.5264
No-fallback mean: 1722.1506323396568
Fallback mean: 2389.770979020979
Difference in means (%): 38.76666385299383


In [ ]:

# Pure GNN event generator
# No fallback.
# Incomplete events are recorded and discarded from complete
# event statistics.


def generate_gnn_events_pure(
    transition_lookup,
    sources,
    n_events=5000,
    event_size=3,
    seed=42
):

    rng = np.random.default_rng(seed)

    sources = np.asarray(
        list(sources),
        dtype=int
    )

    complete_events = []
    incomplete_events = []

    for _ in range(n_events):

        first = int(
            rng.choice(sources)
        )

        event = [first]

        while len(event) < event_size:

            current = event[-1]

            if current not in transition_lookup:
                break

            info = transition_lookup[current]

            # Support dictionary format
            if isinstance(info, dict):

                targets = np.asarray(
                    info["targets"],
                    dtype=int
                )

                probabilities = np.asarray(
                    info["probabilities"],
                    dtype=float
                )

            # Support tuple format
            else:

                targets = np.asarray(
                    info[0],
                    dtype=int
                )

                probabilities = np.asarray(
                    info[1],
                    dtype=float
                )

            # Remove already-selected locations
            mask = ~np.isin(
                targets,
                np.asarray(event)
            )

            targets = targets[mask]
            probabilities = probabilities[mask]

            if (
                len(targets) == 0
                or probabilities.sum() <= 0
            ):
                break

            probabilities = (
                probabilities
                / probabilities.sum()
            )

            next_location = int(
                rng.choice(
                    targets,
                    p=probabilities
                )
            )

            event.append(next_location)

        if len(event) == event_size:
            complete_events.append(event)
        else:
            incomplete_events.append(event)

    return complete_events, incomplete_events

In [ ]:

# Recreate GNN candidate source locations


candidate_sources = (
    gnn_transition_df["breach_i"]
    .drop_duplicates()
    .astype(int)
    .to_numpy()
)

print(
    "Candidate source locations:",
    len(candidate_sources)
)

print(
    "First 10:",
    candidate_sources[:10]
)

Candidate source locations: 569
First 10: [ 14  25 191 205 220 381 386 392 393 429]


In [52]:

# GNN completion rate by event size


completion_rows = []

for size in [2, 3, 4, 5]:

    complete, incomplete = generate_gnn_events_pure(
        gnn_transition,
        candidate_sources,
        n_events=5000,
        event_size=size,
        seed=5000 + size
    )

    completion_rows.append({
        "event_size": size,
        "requested_events": 5000,
        "complete_events": len(complete),
        "incomplete_events": len(incomplete),
        "completion_rate": len(complete) / 5000
    })

completion_results = pd.DataFrame(
    completion_rows
)

display(completion_results)

,event_size,requested_events,complete_events,incomplete_events,completion_rate
0,2,5000,5000,0,1.0000
1,3,5000,4416,584,0.8832
2,4,5000,3519,1481,0.7038
3,5,5000,2565,2435,0.5130


In [53]:

# Handcrafted-kernel completion rate


kernel_completion_rows = []

# Use the original 594-source kernel population
kernel_sources = np.array(
    list(kernel_lookup.keys()),
    dtype=int
)

for size in [2, 3, 4, 5]:

    complete, incomplete = (
        generate_gnn_events_pure(
            # Adapter: convert kernel format to the
            # same tuple format expected by the function
            {
                int(source): (
                    np.asarray(
                        info["targets"],
                        dtype=int
                    ),
                    np.asarray(
                        info["weights"],
                        dtype=float
                    )
                )
                for source, info in kernel_lookup.items()
            },
            kernel_sources,
            n_events=5000,
            event_size=size,
            seed=6000 + size
        )
    )

    kernel_completion_rows.append({
        "event_size": size,
        "requested_events": 5000,
        "complete_events": len(complete),
        "incomplete_events": len(incomplete),
        "completion_rate": len(complete) / 5000
    })

kernel_completion_results = pd.DataFrame(
    kernel_completion_rows
)

display(kernel_completion_results)

,event_size,requested_events,complete_events,incomplete_events,completion_rate
0,2,5000,5000,0,1.0000
1,3,5000,4556,444,0.9112
2,4,5000,3939,1061,0.7878
3,5,5000,3281,1719,0.6562


## Interpretation

The experiments suggest that a graph-based pairwise dependence model can recover strong spatial and system-level clustering in synthetic events. However, the sequential pairwise generator becomes increasingly incomplete as event size grows.

That is the key limitation I take forward: pairwise transition probabilities do not automatically define a coherent higher-order joint event distribution. The result motivates moving from pairwise sequential sampling toward a higher-order probabilistic event model, while keeping hydraulic consequences and climate conditioning as explicit parts of the future framework.
